# Лабораторная работа №4
## Демонстрация работы автоматизированного workflow
**Тема диплома:** Разработка системы неразрушающего контроля для выявления дефектов металлических бутылок на конвейерной линии

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from src.webhook_handler import WorkflowClient
from dotenv import load_dotenv

# Загрузка переменных из docker/.env
load_dotenv(os.path.join('..', 'docker', '.env'))
client = WorkflowClient(base_url='http://localhost:5678')

### 1. Проверка доступности n8n

In [ ]:
status = client.check_status()
if status.get('available'):
    print('✅ n8n доступен')
else:
    print(f'❌ n8n недоступен: {status.get("error")}')

### 2. Базовый workflow: обработка заявки

In [ ]:
result = client.send_application(
    message='Не работает вход в систему, ошибка 403',
    contact='user@test.com'
)
if result['success']:
    print('✅ Заявка обработана')
    print('Ответ AI:', result.get('response', {}))
else:
    print(f'❌ Ошибка: {result.get("error")}')

### 3. Специализированный workflow: контроль качества бутылки

In [ ]:
measurements = 'd=65.1mm, h=169.9mm, wall_min=0.22mm, wall_nom=0.30mm, ect_amplitude=18.7mV, anomaly_flag=1'
result = client.send_inspection_data(measurements, 'BTL-DEMO-001')
if result['success']:
    print('✅ Инспекция выполнена')
    # Ответ от YandexGPT уже внутри response
    print('Ответ workflow:', result.get('response', {}))
else:
    print(f'❌ Ошибка: {result.get("error")}')

### 4. Проверка истории выполнения в PostgreSQL
Запустите в терминале (или в этом ноутбуке через subprocess):

In [ ]:
import subprocess
cmd = 'docker exec -it n8n_postgres psql -U n8n -d n8n -c "SELECT id, status, \"startedAt\" FROM execution_entity ORDER BY \"startedAt\" DESC LIMIT 5;"'
subprocess.run(cmd, shell=True)

### 5. Выводы
- Workflow автоматизирует обработку заявок и контроль качества.
- AI-классификация выполняется корректно.
- Данные фиксируются в PostgreSQL.